# Build Your First Intelligent Agent Team with ADK & LiteLLM
## Addressing Quota Limits with Multi-Model Support

This notebook demonstrates how to build a robust Agent team using **Google ADK** (Agent Development Kit). 

Crucially, it addresses common **API quota limits** (like those with Gemini) by integrating **LiteLLM**. This allows you to seamless switch between or mix different model providers (e.g., **Mistral**, **OpenAI**, **Anthropic**, **MiniMax**) within the same agent system.

### What we will build:
1. **Basic Agent**: A weather agent using a primary model.
2. **Multi-Model Setup**: Configure agents to use Mistral or GPT-4o via `LiteLlm`.
3. **Agent Team**: A Root agent delegating to specialized sub-agents (Greeting, Farewell).
4. **State Management**: Using `Session State` to remember user preferences.
5. **Safety Guardrails**: Blocking specific inputs and tool usage.

### Prerequisites
- Python environment
- API Keys for the models you intend to use (Gemini, Mistral, OpenAI, etc.)

In [ ]:
# @title Step 0: Setup and Installation
!pip install google-adk litellm -q
print("Installation complete.")

In [ ]:
# @title Step 1: Import Libraries & Configure API Keys
import os
import asyncio
import logging
from typing import Optional, Dict, Any

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm  # Essential for multi-model support
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.adk.tools.base_tool import BaseTool
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types 

# Configure Logging
logging.basicConfig(level=logging.ERROR)

# --- CONFIGURATION: API KEYS ---
# Replace with your actual keys. 
# Using os.environ is standard for LiteLLM.

# 1. Gemini (Google)
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_KEY_HERE"

# 2. Mistral AI (Alternative provider for Load Balancing)
os.environ["MISTRAL_API_KEY"] = "YOUR_MISTRAL_KEY_HERE"

# 3. OpenAI (Another alternative)
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY_HERE"

# Configure ADK to use API keys directly
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print("Libraries imported and environment configured.")

In [ ]:
# @title Step 2: Define and Mix Models
# This is where we solve the Quota Limit problem.
# Instead of relying on a single model string, we define multiple providers.

# 1. Standard Gemini Model
MODEL_GEMINI = "gemini-2.0-flash"

# 2. Mistral Model (via LiteLLM)
# LiteLLM format: "provider/model-name"
MODEL_MISTRAL = "mistral/mistral-large-latest" 

# 3. OpenAI Model (via LiteLLM)
MODEL_GPT4O = "openai/gpt-4o"

# 4. MiniMax (via LiteLLM - Anthropic Compatible)
MODEL_MINIMAX = "minimax/MiniMax-M2.1"

print("Model definitions ready.")

In [ ]:
# @title Step 3: Define Tools
# Tools are shared python functions.

def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city.
    
    Args:
        city (str): The name of the city.
    """
    print(f"--> Tool Call: get_weather('{city}')")
    # Mock data
    return {"status": "success", "report": f"It is sunny in {city} with 25C."}

def say_hello(name: Optional[str] = None) -> str:
    """Provides a simple greeting."""
    print(f"--> Tool Call: say_hello('{name}')")
    return f"Hello, {name or 'User'}!"

def say_goodbye() -> str:
    """Provides a farewell message."""
    print(f"--> Tool Call: say_goodbye()")
    return "Goodbye! Have a nice day."

print("Tools defined.")

In [ ]:
# @title Step 4: Create a Hybrid Agent Team
# Here we assign different models to different agents to distribute load.

# 1. Greeting Agent -> Uses Mistral (cheaper/faster for simple tasks)
try:
    greeting_agent = Agent(
        name="greeting_agent",
        model=LiteLlm(model=MODEL_MISTRAL), # <--- USING MISTRAL
        description="Handles greetings.",
        instruction="Greet the user warmly.",
        tools=[say_hello]
    )
    print("Greeting Agent (Mistral) created.")
except Exception as e:
    print(f"Skipping Greeting Agent (Mistral) due to missing key: {e}")
    greeting_agent = None


# 2. Farewell Agent -> Uses Gemini (Standard)
try:
    farewell_agent = Agent(
        name="farewell_agent",
        model=MODEL_GEMINI, # <--- USING GEMINI
        description="Handles farewells.",
        instruction="Say goodbye politely.",
        tools=[say_goodbye]
    )
    print("Farewell Agent (Gemini) created.")
except Exception as e:
    print(f"Skipping Farewell Agent (Gemini): {e}")
    farewell_agent = None

sub_agents_list = []
if greeting_agent: sub_agents_list.append(greeting_agent)
if farewell_agent: sub_agents_list.append(farewell_agent)

# 3. Root Agent -> Uses GPT-4o or a strong Gemini model (Orchestrator)
# If you don't have GPT key, you can swap this back to MDEL_GEMINI
try:
    root_agent = Agent(
        name="root_agent",
        model=LiteLlm(model=MODEL_GPT4O), # <--- USING GPT-4o
        description="Main assistant.",
        instruction="You are the main coordinator. "
                    "Delegate greetings to 'greeting_agent'. "
                    "Delegate farewells to 'farewell_agent'. "
                    "For weather requests, use your own 'get_weather' tool.",
        tools=[get_weather],
        sub_agents=sub_agents_list # <--- DELEGATION
    )
    print(f"Root Agent (GPT-4o) created.")
except Exception as e:
    print(f"Error creating root agent: {e}")
    root_agent = None

In [ ]:
# @title Step 5: Run the Interaction
# Helper function to run the agent

async def run_interaction(agent, query):
    if not agent:
        print("Agent is not initialized. Check API keys.")
        return

    print(f"\n--- User: {query} ---")
    
    session_service = InMemorySessionService()
    session_id = "demo_session"
    user_id = "demo_user"
    
    await session_service.create_session(app_name="demo", user_id=user_id, session_id=session_id)
    
    runner = Runner(agent=agent, app_name="demo", session_service=session_service)
    
    content = types.Content(role="user", parts=[types.Part(text=query)])
    
    try:
        async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
            if event.is_final_response():
                print(f"[{event.author}]: {event.content.parts[0].text if event.content.parts else 'No text'}")
    except Exception as e:
        print(f"Execution Error: {e}")

# Run scenarios
# 1. Greeting (Should use Mistral)
print("Testing Delegation to Mistral...")
await run_interaction(root_agent, "Hello there!")

# 2. Weather (Should use Root Agent directly)
print("\nTesting Root Agent execution...")
await run_interaction(root_agent, "What is the weather in Paris?")

# 3. Farewell (Should use Gemini)
print("\nTesting Delegation to Gemini...")
await run_interaction(root_agent, "Bye bye!")